# RAG con LlamaIndex: Texto (PDF) y Multi-Modal

Notebook condensado que combina los componentes principales de LlamaIndex en dos partes:

**Parte 1 — RAG sobre texto (PDF de un cuento)**
1. Question Answering
2. Summarization
3. ChatEngine (Simple, CondenseQuestion, Context, CondensePlusContext)
4. Customizing QA System
5. Index as Retriever

**Parte 2 — RAG Multi-Modal (imágenes + texto)**
1. Querying images con un LLM Multi-Modal


## Instalación

In [ ]:
!pip install llama-index
!pip install llama-index-readers-file pypdf
!pip install llama-index-llms-openai
!pip install llama-index-embeddings-openai
!pip install llama-index-multi-modal-llms-openai
!pip install llama-index-vector-stores-qdrant
!pip install llama-index-embeddings-clip
!pip install ftfy regex tqdm matplotlib scikit-image
!pip install git+https://github.com/openai/CLIP.git

## Configurar API Key

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

---
# Parte 1 — RAG sobre texto (PDF de un cuento)

Retrieval Augmented Generation


## Cargar el PDF del cuento

In [28]:
from llama_index.core import SimpleDirectoryReader

# Ajusta esta ruta al nombre de tu archivo PDF
PDF_PATH = r"C:\Users\santi\Downloads\el-jardin-de-los-senderos-que-se-bifurcan_borges.pdf"

documents = SimpleDirectoryReader(input_files=[PDF_PATH]).load_data()

print(f"Páginas/documentos cargados: {len(documents)}")
print(f"Vista previa del primer fragmento:\n{documents[0].text[:1000]}...")

Páginas/documentos cargados: 7
Vista previa del primer fragmento:
El jardín de los senderos que se bifurcan 
Jorge Luis Borges 
 
En la página 242 de la Historia de la guerra europea, de Liddell Hart, se lee que 
una ofensiva de trece divisiones británicas (apoyadas por mil cuatrocientas piezas de 
artillería) contra la línea Serre -Montauban había sido planeada para el veinticuatro de 
julio de 1916 y debió postergarse hasta la mañana de l día veintinueve. Las lluvias 
torrenciales (anota el capitán Liddell Hart) provocaron esa demora -nada significativa, 
por cierto-. La siguiente declaración, dictada, releída y firmada por el doctor Yu Tsun, 
antiguo catedrático de inglés en la Hochschule de Tsingtao, arroja una insospechada luz 
sobre el caso. Faltan las dos páginas iniciales. 
“…y colgué el tubo. Inmediatamente después, reconocí la voz que había 
contestado en alemán. Era la del capitán Richard Madden. Madden, en el departamento 
de Viktor Runeberg, quería decir el fin de nuestros

## Configurar LLM y modelo de Embeddings

In [29]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

# gpt-4o-mini reemplaza a gpt-3.5-turbo (deprecado/legacy)
llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

Settings.llm = llm
Settings.embed_model = embed_model

## Crear Nodes (chunking)

In [32]:
from llama_index.core.node_parser import TokenTextSplitter

splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Total de nodos: {len(nodes)}")
nodes[0]

Total de nodos: 19


TextNode(id_='58612134-f455-4566-bc88-da990aa5eeae', embedding=None, metadata={'page_label': '1', 'file_name': 'el-jardin-de-los-senderos-que-se-bifurcan_borges.pdf', 'file_path': 'C:\\Users\\santi\\Downloads\\el-jardin-de-los-senderos-que-se-bifurcan_borges.pdf', 'file_type': 'application/pdf', 'file_size': 245590, 'creation_date': '2026-04-28', 'last_modified_date': '2026-04-28'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ec497f03-fb79-4469-adc6-10ba02269922', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'el-jardin-de-los-senderos-que-se-bifurcan_borges.pdf', 'file_path': 'C:\\Users\\santi\\Downloads\\el-jardin-de-los-senderos-que-se-bifurcan_borges.pdf', 'file_typ

## Crear el Índice Vectorial

In [34]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)

## Crear QueryEngine y consultar

In [35]:
query_engine = index.as_query_engine(similarity_top_k=5)

In [36]:
# Ajusta la pregunta al contenido de tu cuento
response = query_engine.query("¿Quién es el autor del cuento?")
print(response)

El autor del cuento es Jorge Luis Borges.


In [37]:
# Ajusta la pregunta al contenido de tu cuento
response = query_engine.query("¿Donde vive el señor Albert?")
print(response)

El señor Albert vive en una casa que se encuentra lejos de la estación de tren, a la que se puede llegar tomando un camino a la izquierda y doblando a la izquierda en cada encrucijada.


In [38]:
# Ajusta la pregunta al contenido de tu cuento
response = query_engine.query("¿Qué sucede con el protagonista al final del cuento?")
print(response)

Al final del cuento, el protagonista, Yu Tsun, es arrestado por el capitán Richard Madden después de haber asesinado al doctor Stephen Albert. A pesar de haber cumplido su objetivo de comunicar un secreto a Berlín, se siente abrumado por la contrición y el cansancio, y ha sido condenado a la horca. Su victoria es amarga, ya que ha logrado su propósito a un alto costo personal.


In [46]:
# Inspeccionar nodos fuente recuperados
print(f"Número de nodos fuente: {len(response.source_nodes)}")
print(response.source_nodes[3].text)

Número de nodos fuente: 5
años en el Pabellón de la Límpida Soledad. A su muerte, los herederos no encontraron  
sino manuscritos caóticos. La familia, como usted acaso no ignora, quiso adjudicarlos al 
fuego; pero su albacea -un monje taoísta o budista- insistió en la publicación. 
“-Los de la sangre de Ts’ui Pên -repliqué- seguimos execrando a ese monje. Esa 
publicación fue insensata. El libro es un acervo indeciso de borradores contradictorios. 
Lo he examinado alguna vez: en el tercer capítulo muere el héroe, en el cuarto está vivo. 
En cuanto a la otra empresa de Ts’ui Pên, a su Laberinto… 
“-Aquí está el Laberinto -dijo indicándome un alto escritorio laqueado. 
“-¡Un laberinto de marfil! -exclamé-. Un laberinto mínimo… 
“-Un laberinto de símbolos -corrigió-. Un invisible laberinto de tiempo. A mí, 
bárbaro inglés, me ha sido deparado revelar ese misterio diáfano. Al cabo de más de cien 
años, los pormenores son irrecuperables, pero no es difícil conjeturar lo que sucedió. Ts’ui 

## Summarization

In [47]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex(nodes)

summary_query_engine = summary_index.as_query_engine()

summary = summary_query_engine.query("Proporciona un resumen del cuento.")
print(summary)

El cuento narra la historia de Yu Tsun, un espía chino durante la Primera Guerra Mundial, que se encuentra en una situación desesperada tras ser perseguido por el capitán Richard Madden. Tsun tiene un secreto crucial sobre la ubicación de un parque de artillería británico y, al darse cuenta de que su tiempo se agota, decide actuar. A través de una serie de reflexiones sobre el tiempo y la vida, se dirige a la casa de Stephen Albert, un sinólogo que resulta ser un descendiente de su antepasado, Ts’ui Pên, quien había creado un laberinto y un libro que exploran la naturaleza del tiempo y las múltiples posibilidades de la existencia.

En su encuentro, Albert revela que el "jardín de senderos que se bifurcan" es una metáfora del tiempo, donde cada decisión crea múltiples realidades. Sin embargo, en un giro trágico, Tsun, al darse cuenta de que Madden lo está buscando, asesina a Albert para enviar un mensaje a Berlín sobre la ubicación del ataque británico. A pesar de haber logrado su objet

## ChatEngines

### 1. SimpleChatEngine (sin contexto del cuento)

In [50]:
from llama_index.core.chat_engine import SimpleChatEngine

chat_engine = SimpleChatEngine.from_defaults()

response = chat_engine.chat("¿Cuándo ganó el Barcelona su primera Copa de Europa?")
print(response)

El FC Barcelona ganó su primera Copa de Europa, actualmente conocida como la Liga de Campeones de la UEFA, en la temporada 1991-1992. El partido final se disputó el 20 de mayo de 1992 en el Estadio de Wembley, donde el Barcelona venció al Sampdoria por 1-0, gracias a un gol de Ronald Koeman en la prórroga.


### 2. CondenseQuestion ChatEngine

In [52]:
chat_engine = index.as_chat_engine(chat_mode="condense_question", verbose=True)

response = chat_engine.chat("¿Quién es Yu Tsun?")
print(response)

Querying with: ¿Quién es Yu Tsun?
Yu Tsun es un personaje que se encuentra en una situación crítica en la narrativa, donde se ve obligado a tomar decisiones drásticas. Es un agente que, en un momento de desesperación, comete un asesinato para cumplir con un objetivo relacionado con la guerra, revelando un secreto importante. Su acción tiene consecuencias significativas, ya que su condena y el desenlace de su historia están ligados a su decisión de matar a una persona llamada Albert.


In [54]:
response = chat_engine.chat("¿Y a quién asesina al final de la historia?")
print(response)

Querying with: ¿A quién asesina Yu Tsun en la historia?
Yu Tsun asesina a Stephen Albert en la historia.


### 3. Context ChatEngine

In [55]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    memory=memory,
    system_prompt=(
        "Eres un asistente que conversa de forma natural y también puede responder "
        "preguntas sobre el contenido de un cuento."
    ),
)

response = chat_engine.chat("Hola")
print(response)

¡Hola! ¿Cómo puedo ayudarte hoy?


In [56]:
response = chat_engine.chat("¿Cuál es el tema central del cuento?")
print(response)

El tema central de "El jardín de senderos que se bifurcan" de Jorge Luis Borges es la naturaleza del tiempo y la multiplicidad de las realidades. A través de la narrativa, Borges explora la idea de que el tiempo no es lineal, sino que se bifurca en múltiples posibilidades y desenlaces. La obra presenta un laberinto de decisiones y caminos que se entrelazan, lo que refleja la complejidad de la existencia y la interconexión de los eventos. Además, se aborda la relación entre la ficción y la realidad, así como la búsqueda del conocimiento y la comprensión del tiempo como un concepto metafísico. ¿Te gustaría saber más sobre algún aspecto específico?


In [57]:
response = chat_engine.chat("¿Cuál es el nombre del protagonista?")
print(response)

El protagonista de "El jardín de senderos que se bifurcan" se llama Yu Tsun. Es un espía chino que trabaja para Alemania durante la Primera Guerra Mundial. Su historia gira en torno a su misión y las decisiones que toma en un contexto de guerra y traición. Si tienes más preguntas sobre el cuento o el personaje, ¡no dudes en preguntar!


### 4. CondensePlusContext ChatEngine

In [58]:
memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    llm=llm,
    context_prompt=(
        "Eres un asistente que conversa de forma natural y puede comentar sobre un cuento.\n"
        "Aquí están los documentos relevantes para el contexto:\n"
        "{context_str}\n"
        "\nInstrucción: usa el historial de chat o el contexto anterior para ayudar al usuario."
    ),
    verbose=True,
)

response = chat_engine.chat("Cuéntame más sobre el protagonista")
print(response)

Condensed question: Cuéntame más sobre el protagonista
El protagonista de "El jardín de los senderos que se bifurcan" es Yu Tsun, un espía chino que trabaja para Alemania durante la Primera Guerra Mundial. A lo largo del cuento, se revela que Yu Tsun es un hombre atrapado entre su lealtad a su país y su deseo de sobrevivir en un mundo hostil. 

Yu Tsun se presenta como un personaje complejo, marcado por la culpa y la desesperación. A pesar de ser un espía, su motivación no es el patriotismo ciego, sino una mezcla de necesidad personal y la presión de un sistema que lo ha llevado a actuar de manera extrema. Su decisión de asesinar a Stephen Albert, un sinólogo que representa un vínculo con su cultura y su historia, es un acto desesperado para transmitir un mensaje crucial a Berlín sobre una ciudad que debe ser atacada. 

A lo largo del relato, se siente su conflicto interno y su contrición por el acto violento que comete. La historia explora temas como el sacrificio, la identidad y la n

## Customizar el RAG Pipeline

In [59]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

# Retriever configurable
retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

# Synthesizer con modo refine
synthesizer = get_response_synthesizer(response_mode="refine")

# Query engine personalizado
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
)

response = query_engine.query("¿Qué moraleja deja el cuento?")
print(response)

El cuento sugiere que la resignación y la aceptación del destino pueden llevar a una forma de fortaleza. A través de la experiencia del protagonista, se destaca la idea de que, ante situaciones difíciles o atroces, es crucial imaginar y aceptar un futuro irrevocable, lo que puede proporcionar una guía en momentos de incertidumbre. Además, se insinúa que la búsqueda de la felicidad y la aventura puede coexistir con la aceptación de la realidad, incluso en circunstancias sombrías.


## Index as Retriever

In [60]:
retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve("¿Quién es el personaje principal?")

from llama_index.core.response.notebook_utils import display_source_node

for text_node in retrieved_nodes:
    display_source_node(text_node, source_length=500)

**Node ID:** c71a4aa4-88d9-451a-86b8-53518b953936<br>**Similarity:** 0.40253771059227755<br>**Text:** pero ese hombre era fuerte como una estatua, pero ese hombre avanzaba por el sendero y 
era el capitán Richard Madden. 
“-El porvenir ya existe -respondí-, pero yo soy su amigo. ¿Puedo examinar de 
nuevo la carta? 
“Albert se levantó. Alto, abrió el cajón del alto escritorio; me dio por un 
momento la espalda. Yo había preparado el revólver. Disparé con sumo cuidado: Albert 
se desplomó sin una queja, inmediatamente. Yo juro que su muerte fue instantánea: una 
fulminación. 
“Lo demás es irrea...<br>

**Node ID:** 14f178df-7133-426c-a945-ff901a3cf994<br>**Similarity:** 0.3702855481774107<br>**Text:** alemán? Subí a mi cuarto; absurdamente cerré 
la puerta con llave y me tiré de espaldas en la estrecha cama de hierro. En la ventana 
estaban los tejados de siempre y el sol nublado de las seis. Me pareció increíble que ese 
día sin premoniciones ni símbolos fuera el de mi muerte implacable. A pesar de mi padre 
muerto, a pesar de haber sido un niño en un simétrico jardín de Hai Feng, ¿yo, ahora, iba 
a morir? Después reflexioné que todas las cosas que suceden a uno precisamente, 
precisament...<br>

**Node ID:** 055599e6-db83-441d-96d4-293b31d1426f<br>**Similarity:** 0.3661760361972188<br>**Text:** se enfrenta 
con diversas alternativas opta por una y elimina las otras; en la del casi inextricable Ts’ui 
Pên, opta -simultáneamente- por todas. Crea, así, diversos porvenires, diversos tiempos, 
que también proliferan y se bifurcan. De ahí las contradicciones de la novela. Fang, 
digamos, tiene un secreto; un desconocido llama a su puerta; Fang resuelve matarlo. 
Naturalmente, hay varios desenlaces posib les: Fang puede matar al intruso, el intruso 
puede matar a Fang, ambos pueden salvars...<br>

---
# Parte 2 — Multi-Modal RAG System

Construimos un sistema RAG que combina texto e imágenes usando un LLM multi-modal.

## Consultar imágenes desde URL con un LLM multi-modal

In [61]:
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, TextBlock, ImageBlock

# gpt-4o reemplaza a gpt-4-vision-preview (deprecado) y soporta visión nativamente
openai_mm_llm = OpenAI(model="gpt-4o", max_tokens=300)

image_url = "https://res.cloudinary.com/hello-tickets/image/upload/c_limit,f_auto,q_auto,w_1920/v1640835927/o3pfl41q7m5bj8jardk0.jpg"

In [62]:
msg = ChatMessage(
    role="user",
    blocks=[
        TextBlock(text="Describe la imagen como texto alternativo"),
        ImageBlock(url=image_url),
    ],
)

response = openai_mm_llm.chat([msg])
print(response.message.content)

El Coliseo de Roma iluminado por la noche con los colores de la bandera italiana: verde, blanco y rojo. El cielo es azul oscuro y hay algunas nubes visibles. La estructura del Coliseo se destaca con luces cálidas en sus arcos inferiores. En el primer plano, se observa una calle y algunas personas caminando.
